# Distant Listening Corpus → BOPP

This notebook exports the complete **notes** and **extended** (DLC resource name: `expanded`) facets. A BOPP annotation has one timeline, so the complete facets are represented by one annotation per `(corpus, piece)` pair. Each annotation is written as MessagePack, JSON, and CSV, then round-trip validated.

In [1]:
from pathlib import Path

import pandas as pd

from scripts.dlc_to_bopp import FACETS, export_all, get_facet, load_dlc, payload_columns

ROOT = Path.cwd()
OUTPUT_DIR = ROOT / 'dlc_bopp_exports'


C:\Users\hentschel\git\bopp\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Descriptor-derived payload schemata

The reusable helpers derive payload fields from the DLC datapackage descriptor: all non-identifier/non-extent columns for notes, and all columns beginning with `label` for the extended harmony facet.

In [2]:
descriptor, package = load_dlc(ROOT)
for facet in FACETS:
    spec, fields, frame = get_facet(descriptor, package, facet)
    print(f'{facet}: {len(frame):,} rows; {len(payload_columns(spec, fields))} payload columns')
    print(payload_columns(spec, fields))


notes: 1,688,754 rows; 21 payload columns
['mc', 'mn', 'quarterbeats', 'mc_onset', 'mn_onset', 'timesig', 'staff', 'voice', 'duration', 'gracenote', 'nominal_duration', 'scalar', 'tied', 'tpc', 'midi', 'name', 'octave', 'chord_id', 'tremolo', 'volta', 'tuning']
extended: 242,867 rows; 23 payload columns
['label', 'globalkey', 'localkey', 'pedal', 'chord', 'special', 'numeral', 'form', 'figbass', 'changes', 'relativeroot', 'cadence', 'phraseend', 'chord_type', 'globalkey_is_minor', 'localkey_is_minor', 'chord_tones', 'added_tones', 'root', 'bass_note', 'alt_label', 'volta', 'placement']


## Complete exports

Run this cell to create `dlc_bopp_exports/notes/` and `dlc_bopp_exports/extended/`. Each contains all pieces in all three formats. MessagePack and JSON are validated byte-for-byte for every piece; a representative CSV round trip is validated for each facet (CSV remains inherently untyped).

In [3]:
sizes, comparison = export_all(ROOT, OUTPUT_DIR)
comparison


Successfully saved to C:\Users\hentschel\git\bopp\dlc_bopp_exports\notes\ABC__n01op18-1_01.bopp.msgpack (461369 bytes)
Successfully saved to C:\Users\hentschel\git\bopp\dlc_bopp_exports\notes\ABC__n01op18-1_01.bopp.json (493463 bytes)
Successfully saved to C:\Users\hentschel\git\bopp\dlc_bopp_exports\notes\ABC__n01op18-1_02.bopp.msgpack (292057 bytes)
Successfully saved to C:\Users\hentschel\git\bopp\dlc_bopp_exports\notes\ABC__n01op18-1_02.bopp.json (320336 bytes)
Successfully saved to C:\Users\hentschel\git\bopp\dlc_bopp_exports\notes\ABC__n01op18-1_03.bopp.msgpack (125204 bytes)
Successfully saved to C:\Users\hentschel\git\bopp\dlc_bopp_exports\notes\ABC__n01op18-1_03.bopp.json (127542 bytes)
Successfully saved to C:\Users\hentschel\git\bopp\dlc_bopp_exports\notes\ABC__n01op18-1_04.bopp.msgpack (546630 bytes)
Successfully saved to C:\Users\hentschel\git\bopp\dlc_bopp_exports\notes\ABC__n01op18-1_04.bopp.json (682853 bytes)
Successfully saved to C:\Users\hentschel\git\bopp\dlc_bopp_e

ValidationError: Could not convert '' with type str: tried to convert to double - at `$.payload.quarterbeats`

## Format-size comparisons

`format_size_comparison.csv` reports complete-facet totals and average storage per observation; `format_sizes_by_piece.csv` keeps the per-piece data for downstream analysis.

In [ ]:
comparison.pivot(index='facet', columns='format', values='total_bytes').assign(
    json_vs_msgpack=lambda table: table['json'] / table['msgpack'],
    csv_vs_msgpack=lambda table: table['csv'] / table['msgpack'],
)
